# 02 · Preprocessing


Clean the data and stratify by churn to keep a similar class balance across splits. The customer ID stays in the split files for overlap checks.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from src.ingestion import read_data, clean_data, split_data
from src.config import CUSTOMER_ID, TARGET

splits = split_data(clean_data(read_data()))
for name, frame in splits.items():
    print(name, len(frame), f"{frame[TARGET].mean():.1%} churn")
all_ids = [customer for frame in splits.values() for customer in frame[CUSTOMER_ID]]
assert len(all_ids) == len(set(all_ids))

Fit medians, most frequent categories, scaling and one-hot levels on training only. Reuse the same rules on validation and at inference.

In [ ]:
from src.preprocessing import fit_preprocessor, transform

preprocessor = fit_preprocessor(splits["train"])
x_train = transform(preprocessor, splits["train"])
x_validation = transform(preprocessor, splits["validation"])
print(x_train.shape, x_validation.shape, x_train.dtype)
print(preprocessor.get_feature_names_out()[:10])